In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json
import itertools
from collections import Counter
# For visualization
import plotly.express as px
import matplotlib.colors as mcolors
import plotly.graph_objects as go


In [3]:
df_topic_taxonomy = pd.read_excel('../data/processed/taxonomy.xlsx')

Merge labels in phase 3 (so label becomes subcat)

In [ ]:
df_label2subcatcat = pd.read_excel('../data/processed/label2subcatcat.xlsx')
df_label2subcat = df_label2subcatcat[df_label2subcatcat['Sub-Category'].notnull()][['GPT Label', 'Sub-Category']]
df_label2subcat

,GPT Label,Sub-Category
31,Pushback Against Boundaries,Social Deterioration and Relationship Strain
32,Communication Tension and Breakdown,Social Deterioration and Relationship Strain
33,Social Isolation and Limited Friendships,Social Deterioration and Relationship Strain
34,Family Estrangement and Disconnection,Social Deterioration and Relationship Strain
35,Unfaithful Apology,Interpersonal Harm and Moral Disregard
...,...,...
102,Michelle Obama Being a Man,Defamatory Conspiracy Targeting Public Figures
103,Tax Evasion,Policy Defiance
104,Abortion Opposition,Policy Defiance
105,China and Global Influence Concerns,Anti-Communism and Geopolitical Distrust


In [6]:
label2subcat = dict(df_label2subcat.values)

In [8]:
def check_subcat(label):
    # Use subcat if given
    if label in label2subcat.keys():
        return label2subcat[label]
    else: return label

df_topic_taxonomy['Label'] = df_topic_taxonomy.apply(lambda row: check_subcat(row['GPT Label']), axis=1)

In [ ]:
df_topic_taxonomy.to_csv('../data/processed/taxonomy_w_label.csv', index=False)

In [40]:
topic2label = dict()
for index, row in df_topic_taxonomy.iterrows():
    for topic in str(row['Topic']).split(','):
        topic2label[int(topic)] = row['Label']

In [41]:
topic2label

{4: 'Interpersonal Harm and Moral Disregard',
 5: 'Social Deterioration and Relationship Strain',
 8: 'COVID-19 Lockdowns',
 20: 'Physical Health Disorders',
 40: 'Child Harm and Elite Abuse Narratives',
 42: 'Social Deterioration and Relationship Strain',
 43: 'Experiences of Race and Racism',
 44: 'Social Deterioration and Relationship Strain',
 45: 'Medical Mistrust',
 50: 'Lifestyle Changes Driven by Conspiratorial Beliefs',
 52: 'Social Deterioration and Relationship Strain',
 53: 'Substance Use and Abuse',
 54: 'Social Deterioration and Relationship Strain',
 56: 'Social Deterioration and Relationship Strain',
 73: 'YouTube Algorithms',
 78: 'COVID-19 Lockdowns',
 327: 'COVID-19 Lockdowns',
 83: '(Un)employment and Retirement Status',
 86: 'Epistemic Rigidity and Distrust in Expertise',
 91: 'Defamatory Conspiracy Targeting Public Figures',
 92: 'Conservative Political Identity',
 99: 'Radical Ideological Alignment and Intolerance',
 102: 'Anti-Government and Institutional Distru

Match sentences to topic (alr did)

In [ ]:
df_sent2topic = pd.read_csv('../data/processed/sent2topic.csv')
df_sent2topic

,Sentence,Topic
0,\tI've known about this group for a while and ...,68
1,My ex-wife went down the path of QANON startin...,-1
2,I was too late to catch it.,-1
3,I didn't even know what QANON was before summe...,0
4,"What an evil, horrible and vile think so many ...",270
...,...,...
235689,She said that when the govt turns all the red ...,-1
235690,I mean it’s just crazy town.,-1
235691,She started the pandemic saying the virus was ...,-1
235692,"Eventually it reached the rural areas, & then ...",32


Match the categorization with the sents to posts

In [43]:
def get_stage(label):
    return df_topic_taxonomy[df_topic_taxonomy['Label']==label]['Stage'].values[0]

In [ ]:
df_post = pd.read_pickle('../data/oon/df_post_oon.pkl')
df_post['sentences_topics'] = pd.Series()
df_post['sentences_labels'] = pd.Series()
df_post['Label'] = pd.Series()

from nltk.tokenize import sent_tokenize
def remove_short_sentences(sentences, min_tok=5):
  return [sent for sent in sentences if len(sent.split()) >= min_tok]
df_post["sentences"] = df_post["selftext"].apply(sent_tokenize)
df_post["sentences"] = df_post["sentences"].apply(remove_short_sentences)
sent_idx = 0

for post_idx, sentence_list in df_post["sentences"].items():
    topics = [df_sent2topic._get_value(i, 'Topic') for i in range(sent_idx, sent_idx + len(sentence_list))]
    df_post.at[post_idx, 'sentences_topics'] = topics
    labels = [topic2label[topic] for topic in topics if topic in topic2label.keys()]
    df_post.at[post_idx, 'sentences_labels'] = labels
    
    labels = list(set(labels)) # To make values unique

    # If no trigger (stage 2) if found but stage 1&3 is found, add 'Unknown' trigger
    label_stages = set([get_stage(label) for label in labels])
    if (len(labels) != 0) and ({1,3}.issubset(label_stages) and (not 2 in label_stages)):
        labels.append('Unknown')
    
    df_post.at[post_idx, 'Label'] = labels
    sent_idx += len(sentence_list)

In [45]:
post2labels = df_post['Label'].to_dict()

In [46]:
list(post2labels.items())[:5]

[(0, ['Social Deterioration and Relationship Strain']),
 (1,
  ['Child Harm and Elite Abuse Narratives',
   'Social Deterioration and Relationship Strain']),
 (2, ['Social Deterioration and Relationship Strain']),
 (3,
  ['Asian and Immigrant Family Dynamics',
   'Interpersonal Harm and Moral Disregard',
   'Unknown']),
 (4,
  ['YouTube Algorithms',
   'Child Harm and Elite Abuse Narratives',
   'Radical Ideological Alignment and Intolerance',
   'COVID-19 Lockdowns'])]

Match label to cat (for coloring in the Sankey)

In [47]:
label2cat = dict([(row['GPT Label'], row['Category']) if pd.isnull(row['Sub-Category']) else (row['Sub-Category'], row['Category']) for _, row in df_label2subcatcat.iterrows()])

In [48]:
label2cat['Unknown'] = 'Unknown'

Labels co-occurance

In [ ]:
edges = []

for labels in post2labels.values():
    if len(labels) >= 2:
        for label1, label2 in itertools.combinations(labels, 2):
            stage1 = get_stage(label1)
            stage2 = get_stage(label2)
            
            # Determine correct direction based on stage
            if stage1 <= stage2:
                edges.append((label1, label2))
            else:
                edges.append((label2, label1))

Sankey

In [ ]:
# Count co-occurrences
edge_counts = Counter(edges)

# Format for Sankey: source, target, and value
sankey_data = pd.DataFrame(edge_counts.items(), columns=['pair', 'value'])
sankey_data[['source', 'target']] = pd.DataFrame(sankey_data['pair'].tolist(), index=sankey_data.index)
sankey_data = sankey_data[['source', 'target', 'value']]

sankey_data.head()

,source,target,value
0,Child Harm and Elite Abuse Narratives,Social Deterioration and Relationship Strain,530
1,Asian and Immigrant Family Dynamics,Interpersonal Harm and Moral Disregard,17
2,Asian and Immigrant Family Dynamics,Unknown,39
3,Unknown,Interpersonal Harm and Moral Disregard,389
4,YouTube Algorithms,Child Harm and Elite Abuse Narratives,49


In [51]:
# Filter connections with too little value
sankey_data = sankey_data[sankey_data["value"] >= 2]

In [52]:
def get_cat(label):
    return label2cat[label]

In [53]:
sankey_data["source_stage"] = sankey_data["source"].apply(get_stage)
sankey_data["target_stage"] = sankey_data["target"].apply(get_stage)

# Keep only source < target to ensure left-to-right flow
sankey_data = sankey_data[sankey_data["target_stage"] - sankey_data["source_stage"] == 1] # To make it strictly consequential

In [54]:
# Sort labels by Stage → Subcategory → Label
unique_labels = sorted(
    set(sankey_data["source"].tolist() + sankey_data["target"].tolist()),
    key=lambda x: (get_stage(x), get_cat(x), x)
)

# Map labels to indices
label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}

In [55]:
# Add indices to DataFrame
sankey_data["source_idx"] = sankey_data["source"].map(label_to_idx)
sankey_data["target_idx"] = sankey_data["target"].map(label_to_idx)

In [56]:
sankey_data.head()

,source,target,value,source_stage,target_stage,source_idx,target_idx
2,Asian and Immigrant Family Dynamics,Unknown,39,1,2,15,29
3,Unknown,Interpersonal Harm and Moral Disregard,389,2,3,29,32
4,YouTube Algorithms,Child Harm and Elite Abuse Narratives,49,2,3,28,41
5,YouTube Algorithms,Radical Ideological Alignment and Intolerance,53,2,3,28,36
8,COVID-19 Lockdowns,Child Harm and Elite Abuse Narratives,279,2,3,19,41


In [57]:
idx_to_label = {idx: label for idx, label in enumerate(unique_labels)}
def get_label(idx):
    return idx_to_label[idx]

In [59]:
# Define your desired opacity level
opacity = 0.6

# Base color palette
color_palette = px.colors.qualitative.Set3

# Ensure consistent ordering
unique_cats = sorted(set(label2cat.values()))

def rgb_string_to_rgba(rgb_string, opacity):
    # Input like 'rgb(141,211,199)'
    nums = rgb_string.strip()[4:-1]  # Remove 'rgb(' and ')'
    r, g, b = map(int, nums.split(','))
    return f'rgba({r}, {g}, {b}, {opacity})'

category_color_map = {
    cat: rgb_string_to_rgba(color_palette[i % len(color_palette)], opacity)
    for i, cat in enumerate(unique_cats)
}

# Assign node colors
node_colors = [
    category_color_map[get_cat(label)]
    for label in unique_labels
]

In [60]:
# Assign stage to all unique labels
label_x = [0.01 if get_stage(label) == 1 else 0.5 if get_stage(label) == 2 else 0.99 for label in unique_labels]

In [61]:
# Compute total value for each label
label_value = defaultdict(float)
for _, row in sankey_data.iterrows():
    label_value[row['source']] += row['value']
    label_value[row['target']] += row['value']

# Manually assign y values based on stacking
stage_offsets = {1: 0.01, 2: 0.01, 3: 0.01}
stage_heights = defaultdict(float)

label_y = {}
stage_cat_order = defaultdict(list)

for label in unique_labels:
    stage = get_stage(label)
    label_y[label] = stage_offsets[stage]
    # Normalize later, just accumulate first
    stage_offsets[stage] += label_value[label]
    stage_heights[stage] += label_value[label]

# Normalize y values so they're within (0, 1)
for label in unique_labels:
    stage = get_stage(label)
    label_y[label] = label_y[label] / (stage_heights[stage]+0.01)  # normalize within stage

label_y[label] = 0.01 + 0.98 * (label_y[label] / (stage_heights[stage] + 0.01))

# Convert to list ordered by unique_labels
node_y = [label_y[label] for label in unique_labels]

In [62]:
# Create Sankey diagram
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=15,
        thickness=20,
        label=unique_labels,
        x=label_x,
        y=node_y,
        color=node_colors,
    ),
    link=dict(
        source=sankey_data["source_idx"],
        target=sankey_data["target_idx"],
        value=sankey_data["value"],
        # Optional: link colors (could use same subcat coloring)
        color=[category_color_map[get_cat(source)] for source in sankey_data["source"]]
    )
)])

fig.update_layout(
    font_size=12,
    height=900,  # increase if needed
    margin=dict(l=10, r=10, t=40, b=10)
)
fig.show()

In [ ]:
df_post.to_pickle('../data/oon/df_post_oon.pkl')